# Variation par ratio sur l'effectif total

Dans ce notebook est réalisé l'analyse de sensibiltié sur les variations par ratio sur l'effectif total.

## 1 - Environnement

### 1.1 - Dépendances

In [ ]:
# pip

%pip install --upgrade pip

In [ ]:
# installation des dépendances

%pip install --upgrade sys

%pip install --upgrade pathlib

%pip install --upgrade importlib

%pip install --upgrade numpy
%pip install --upgrade pandas

%pip install --upgrade ast

%pip install --upgrade logging

In [ ]:
# chargement des dépendances

import sys

from pathlib import Path

import importlib

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd

import ast

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import logging

In [ ]:
# configuration du logging

logging.basicConfig(
    level=logging.INFO,  
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

In [ ]:
# chemin relatif depuis ce notebook jusqu'au dossier parent de modules 

project_root = Path.cwd().parents[3]
sys.path.insert(0, str(project_root))

### 1.2 - Classes

In [ ]:
# chargement des modules

import modules.CopertEstimator
import modules.DemandVariation
import modules.DemandVariationCarSimulation1Res
import modules.DemandVariationUtils as DemandVariationUtils
import modules.Sensitivity
import modules.SensitivityUtils as SensitivityUtils
importlib.reload(SensitivityUtils)

importlib.reload(sys.modules['modules.CopertEstimator'])
importlib.reload(sys.modules['modules.DemandVariation'])
importlib.reload(sys.modules['modules.DemandVariationCarSimulation1Res'])
importlib.reload(DemandVariationUtils)
importlib.reload(sys.modules['modules.Sensitivity'])

CopertEstimator = modules.CopertEstimator.CopertEstimator
DemandVariation = modules.DemandVariation.DemandVariation
DemandVariationCarSimulation1Res = modules.DemandVariationCarSimulation1Res.DemandVariationCarSimulation1Res
Sensitivity = modules.Sensitivity.Sensitivity

## 2 - Contexte, chargement, prétraitement

### 2.1 - Contexte

l'analyse de sensibilité est d'abords réalisée dans dans le cas le plus simple : l'effectif total de chaque demande est modifié selon un ratio, à chacune de ces variations sont associées des estimations de production d'émissions de NOx et CO2, puis une analyse est menée pour déterminer la manière dont évoluent ces calculs en fonction des modification de l'effectif total. Au terme de cette analyse seront obtenus des indicateurs pour chaque demande, indicateurs qui seront comparés pour comprendre les différences de sensibilité de chaque demande vis à vis de la modification de leur effectif total.

### 2.2 - Chargement et prétraitement

In [ ]:
# demandes

chemin_matsim_calibration = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original/Matsim_7-9_Calibration__Original__None_None__None_None__None__1.00__1_1.00.csv"
matsim_calibration = pd.read_csv(chemin_matsim_calibration, sep=';')

chemin_amsterdam_originale = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Original/AmsterdamCity_7-9_Original__Original__None_None__None_None__None__1.00__1_1.00.csv"
amsterdam_originale = pd.read_csv(chemin_amsterdam_originale, sep=';')

## 3 - Variations

Cette sous partie présente le traitement de chaque demande et l'obtention pour chacune d'entre elles de variations à partir de modification de leur effectif total via ratio. Les modifications sont réalisées par tirage aléatoire uniforme sur l'interval [0.8, 1.2] avec un pas de 0.02.

In [ ]:
# instanciation d'objets Demandvariation

DM_matsim_calibration = DemandVariation(original_demand_path=chemin_matsim_calibration)
DM_amsterdam_originale = DemandVariation(original_demand_path=chemin_amsterdam_originale)

### 3.1 - Variations de la demande Matsim_7-9_Calibration

In [ ]:
# variations totale par ratio

DM_matsim_calibration.apply_total_ratio_variation( 
                            origin="Matsim_7-9_Calibration", 
                            method="WeightedSampling", 
                            law="Uniform", 
                            parameters=None, 
                            interval=[0.6,1.6,0.02], 
                            number_per_ratio=1, 
                            random_state=42, 
                            path="../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Variations__Total-Ratio", 
                            car_only=False)

### 3.2 - Variations de la demande AmsterdamCity_Original

In [ ]:
# variations totale par ratio

DM_amsterdam_originale.apply_total_ratio_variation( 
                            origin="Amsterdam_7-9_Original", 
                            method="WeightedSampling", 
                            law="Uniform", 
                            parameters=None, 
                            interval=[0.6,1.6,0.02], 
                            number_per_ratio=1, 
                            random_state=42, 
                            path="../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Variations__Total-Ratio", 
                            car_only=False)

## 4 - Simulations

Ici sont réalisées les simulations des variations précédemment produites. les simulations sont mises en place avec MnMS pour un fichier graphe provenant d'un précédent travail sur Amsterdam avec le simulateur MATSim et avec des fichiers couches origine-destination et de transit crées avec les modules d'MnMS. Les variables paramètres de la fonction MFD proviennent du travail d'un ancien stagiaire. La simulation est réalisée avec le modèle de décision *DummyDecisionModel*, un modèle déterministe. 

### 4.1 - Simulations pour les variations de la demande Matsim_7-9_Calibration

In [ ]:
# fichiers reseaux

graph_path = "../../network/graph/Amsterdam__Matsim__graph.json"
od_layer_path = "../../network/test_files/Amsterdam__Matsim__graph_od_layer_14_16_1000.json"
transit_link_path = "../../network/test_files/Amsterdam__Matsim__graph_transit_link_14_16_1000_grid.json"
a = 2.8981940480441857
b = -0.00010553060526140915
MFD_variables = [a, b]

# demandes

chemin_demande_matsim_variations = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Variations__Total-Ratio"

In [ ]:
# simulations pour la demande MATSim calibrée

output_directory = "../../outputs/Matsim_7-9/simulations/Matsim_7-9_Calibration__Variations__Total-Ratio__Simulations_Dummy__14_16_1000"

DVCS1R = DemandVariationCarSimulation1Res(chemin_demande_matsim_variations, graph_path)
DVCS1R.simulate(od_layer_path, transit_link_path, MFD_variables, "Dummy", output_directory)

### 4.2 - Simulations pour les variations de la demande AmsterdamCity_Original

In [ ]:
# fichiers reseaux

graph_path = "../../network/graph/Amsterdam__Matsim__graph.json"
od_layer_path = "../../network/test_files/Amsterdam__Matsim__graph_od_layer_14_16_1000.json"
transit_link_path = "../../network/test_files/Amsterdam__Matsim__graph_transit_link_14_16_1000_grid.json"
a = 2.8981940480441857
b = -0.00010553060526140915
MFD_variables = [a, b]

# demandes

chemin_demande_amsterdam_variations = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Variations__Total-Ratio"

In [ ]:
# simulations pour la demande Amsterdam originale

output_directory = "../../outputs/AmsterdamCity_7-9/simulations/AmsterdamCity_7-9_Original__Variations__Total-Ratio__Simulations_Dummy__14_16_1000"

DVCS1R = DemandVariationCarSimulation1Res(chemin_demande_amsterdam_variations, graph_path)
DVCS1R.simulate(od_layer_path, transit_link_path, MFD_variables, "Dummy", output_directory)

## 5 - Copert

L'estimation des émissions via Copert est effectué avec la méthode DISTANCE_DIFF (calcul à partir de la distance parcourue) pour un pas de temps automatique (le trajet total de chaque véhicule est l'unité de base pour l'estimation des émissions).

In [ ]:
S = Sensitivity()

## 5.1 - Tableaux

#### 5.1.1 - Estimation Copert pour les variations de la demande Matsim_7-9_Calibration

In [ ]:
# estimation copert pour les variations de la demande Matsim_7-9_Calibration


original_outputs_directory = "../../outputs/Matsim_7-9/simulations/Matsim_7-9_Calibration__Original__Simulation_Dummy__14_16_1000/Matsim_7-9_Calibration__Original__None_None__None_None__None__1.00__1_1.00"
simulation_outputs_directory = "../../outputs/Matsim_7-9/simulations/Matsim_7-9_Calibration__Variations__Total-Ratio__Simulations_Dummy__14_16_1000"
crs = "epsg:32631"
copert_data_path = "../../data/copert_data/test/FE_COPERT2025.csv"
time_period = "auto"
method = "DISTANCE_DIFF"

S.create_sensitivity_df_from_simulation_outputs_directory(original_outputs_directory=original_outputs_directory,
                                                                         simulation_outputs_directory=simulation_outputs_directory, 
                                                                         period=None, 
                                                                         area=None, 
                                                                         mobility_services=None, 
                                                                         crs=crs, 
                                                                         copert_data_path=copert_data_path, 
                                                                         time_period=time_period, 
                                                                         method=method)

matsim_sensitivity_df = S.get_sensitivity_df()
matsim_sensitivity_df.to_csv("../../outputs/Matsim_7-9/analysis/Matsim_7-9_Calibration__Variations__Total-Ratio__Sensitivity_Dummy__14_16_1000/sensitivity.csv", sep=';')

#### 5.1.2 - Estimation Copert pour les variations de la demande AmsterdamCity_Original

In [ ]:
# estimation copert pour les variations de la demande Amsterdam_7-9_Original


original_outputs_directory = "../../outputs/AmsterdamCity_7-9/simulations/AmsterdamCity_7-9_Original__Original__Simulation_Dummy__14_16_1000/AmsterdamCity_7-9_Original__Original__None_None__None_None__None__1.00__1_1.00"
simulation_outputs_directory = "../../outputs/AmsterdamCity_7-9/simulations/AmsterdamCity_7-9_Original__Variations__Total-Ratio__Simulations_Dummy__14_16_1000"
crs = "epsg:32631"
copert_data_path = "../../data/copert_data/test/FE_COPERT2025.csv"
time_period = "auto"
method = "DISTANCE_DIFF"

S.create_sensitivity_df_from_simulation_outputs_directory(original_outputs_directory=original_outputs_directory,
                                                                         simulation_outputs_directory=simulation_outputs_directory, 
                                                                         period=None, 
                                                                         area=None, 
                                                                         mobility_services=None, 
                                                                         crs=crs, 
                                                                         copert_data_path=copert_data_path, 
                                                                         time_period=time_period, 
                                                                         method=method)

amsterdam_sensitivity_df = S.get_sensitivity_df()
amsterdam_sensitivity_df.to_csv("../../outputs/AmsterdamCity_7-9/analysis/AmsterdamCity_7-9_Original__Variations__Total-Ratio__Simulations_Dummy__14_16_1000/sensitivity.csv", sep=';')

## 5.2 - Régression linéaire

Dans cette sous-partie sont produits des régréssions linéaires à partir des estimations réalisés pour les variations de chaque demande. Les régréssions sont simples avec comme variable explicative le facteur multiplicatif d'obtention des variations et comme à expliquer une des estimation. L'objectif est d'obtenir des coefficients à comparer pour chaque type de demande.

#### 5.2.1 - Affichage

##### 5.2.1.1 - Matsim_7-9_Calibration

In [ ]:
# NOx

matsim_sensitivity_df = pd.read_csv("../../outputs/Matsim_7-9/analysis/Matsim_7-9_Calibration__Variations__Total-Ratio__Sensitivity_Dummy__14_16_1000/sensitivity.csv", sep=';')
SensitivityUtils.plot_target_vs_total_ratio(matsim_sensitivity_df, "NOX_RATIO")

In [ ]:
# CO2

matsim_sensitivity_df = pd.read_csv("../../outputs/Matsim_7-9/analysis/Matsim_7-9_Calibration__Variations__Total-Ratio__Sensitivity_Dummy__14_16_1000/sensitivity.csv", sep=';')
SensitivityUtils.plot_target_vs_total_ratio(matsim_sensitivity_df, "CO2_RATIO")

##### 5.2.1.2 - Amsterdam_7-9_Original

In [ ]:
# NOx

amsterdam_sensitivity_df = pd.read_csv("../../outputs/AmsterdamCity_7-9/analysis/AmsterdamCity_7-9_Original__Variations__Total-Ratio__Simulations_Dummy__14_16_1000/sensitivity.csv", sep=';')
SensitivityUtils.plot_target_vs_total_ratio(amsterdam_sensitivity_df, "NOX_RATIO")

In [ ]:
# CO2

amsterdam_sensitivity_df = pd.read_csv("../../outputs/AmsterdamCity_7-9/analysis/AmsterdamCity_7-9_Original__Variations__Total-Ratio__Simulations_Dummy__14_16_1000/sensitivity.csv", sep=';')
SensitivityUtils.plot_target_vs_total_ratio(amsterdam_sensitivity_df, "CO2_RATIO")

#### 5.2.2 - Application

##### 5.2.2.1 - Matsim_7-9_Calibration

In [ ]:
# régression  Matsim_7-9_Calibration NOx

matsim_sensitivity_df = pd.read_csv("../../outputs/Matsim_7-9/analysis/Matsim_7-9_Calibration__Variations__Total-Ratio__Sensitivity_Dummy__14_16_1000/sensitivity.csv", sep=';')

X = matsim_sensitivity_df[["TOTAL_RATIO"]]  # variables explicatives
y = matsim_sensitivity_df["NOX_RATIO"]

model = LinearRegression()
model.fit(X, y)

coeffs = pd.DataFrame({
    "Variable": X.columns,
    "Coefficient": model.coef_
})

print(coeffs)
print("Intercept :", model.intercept_)



y_pred = model.predict(X)


mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)

print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAE  : {mae:.2f}")
print(f"R²   : {r2:.3f}")




print("Intercept (β0) :", model.intercept_)
print("Coefficient (β1) :", model.coef_[0])

# Plot
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color="steelblue", alpha=0.7, label="Observations")
plt.plot(X, y_pred, color="darkred", linewidth=2, label="Régression linéaire")

plt.xlabel("TOTAL_RATIO")
plt.ylabel("NOX_RATIO")
plt.title("Sensibilité NOx en fonction du TOTAL_RATIO")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# régression  Matsim_7-9_Calibration CO2

matsim_sensitivity_df = pd.read_csv("../../outputs/Matsim_7-9/analysis/Matsim_7-9_Calibration__Variations__Total-Ratio__Sensitivity_Dummy__14_16_1000/sensitivity.csv", sep=';')

X = matsim_sensitivity_df[["TOTAL_RATIO"]]  # variables explicatives
y = matsim_sensitivity_df["CO2_RATIO"]

model = LinearRegression()
model.fit(X, y)

coeffs = pd.DataFrame({
    "Variable": X.columns,
    "Coefficient": model.coef_
})

print(coeffs)
print("Intercept :", model.intercept_)



y_pred = model.predict(X)


mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)

print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAE  : {mae:.2f}")
print(f"R²   : {r2:.3f}")




print("Intercept (β0) :", model.intercept_)
print("Coefficient (β1) :", model.coef_[0])

# Plot
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color="steelblue", alpha=0.7, label="Observations")
plt.plot(X, y_pred, color="darkred", linewidth=2, label="Régression linéaire")

plt.xlabel("TOTAL_RATIO")
plt.ylabel("CO2_RATIO")
plt.title("Sensibilité CO2 en fonction du TOTAL_RATIO")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

##### 5.2.2.1 - Amsterdam_7-9_Original

In [ ]:
# régression  Amsterdam_7-9_Original NOx

amsterdam_sensitivity_df = pd.read_csv("../../outputs/AmsterdamCity_7-9/analysis/AmsterdamCity_7-9_Original__Variations__Total-Ratio__Simulations_Dummy__14_16_1000/sensitivity.csv", sep=';')

X = amsterdam_sensitivity_df[["TOTAL_RATIO"]]  # variables explicatives
y = amsterdam_sensitivity_df["NOX_RATIO"]

model = LinearRegression()
model.fit(X, y)

coeffs = pd.DataFrame({
    "Variable": X.columns,
    "Coefficient": model.coef_
})

print(coeffs)
print("Intercept :", model.intercept_)



y_pred = model.predict(X)


mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)

print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAE  : {mae:.2f}")
print(f"R²   : {r2:.3f}")




print("Intercept (β0) :", model.intercept_)
print("Coefficient (β1) :", model.coef_[0])

# Plot
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color="steelblue", alpha=0.7, label="Observations")
plt.plot(X, y_pred, color="darkred", linewidth=2, label="Régression linéaire")

plt.xlabel("TOTAL_RATIO")
plt.ylabel("NOX_RATIO")
plt.title("Sensibilité NOx en fonction du TOTAL_RATIO")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# régression  Amsterdam_7-9_Original CO2

amsterdam_sensitivity_df = pd.read_csv("../../outputs/AmsterdamCity_7-9/analysis/AmsterdamCity_7-9_Original__Variations__Total-Ratio__Simulations_Dummy__14_16_1000/sensitivity.csv", sep=';')

X = amsterdam_sensitivity_df[["TOTAL_RATIO"]]  # variables explicatives
y = amsterdam_sensitivity_df["CO2_RATIO"]

model = LinearRegression()
model.fit(X, y)

coeffs = pd.DataFrame({
    "Variable": X.columns,
    "Coefficient": model.coef_
})

print(coeffs)
print("Intercept :", model.intercept_)



y_pred = model.predict(X)


mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)

print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAE  : {mae:.2f}")
print(f"R²   : {r2:.3f}")




print("Intercept (β0) :", model.intercept_)
print("Coefficient (β1) :", model.coef_[0])

# Plot
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color="steelblue", alpha=0.7, label="Observations")
plt.plot(X, y_pred, color="darkred", linewidth=2, label="Régression linéaire")

plt.xlabel("TOTAL_RATIO")
plt.ylabel("NOX_RATIO")
plt.title("Sensibilité CO2 en fonction du TOTAL_RATIO")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

##### 5.2.2.3 - Comparatif

In [ ]:
# comparatif NOx

matsim_sensitivity_df = pd.read_csv("../../outputs/Matsim_7-9/analysis/Matsim_7-9_Calibration__Variations__Total-Ratio__Sensitivity_Dummy__14_16_1000/sensitivity.csv", sep=';')

X = matsim_sensitivity_df[["TOTAL_RATIO"]]  # variables explicatives
y = matsim_sensitivity_df["NOX_RATIO"]

model = LinearRegression()
model.fit(X, y)

y_pred = model.predict(X)

# Plot
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color="red", alpha=0.4,s=10, label="Matsim_7-9_Calibration  Observations")
plt.plot(X, y_pred, color="darkred", linewidth=0.8, label="Matsim_7-9_Calibration Régression linéaire")


amsterdam_sensitivity_df = pd.read_csv("../../outputs/AmsterdamCity_7-9/analysis/AmsterdamCity_7-9_Original__Variations__Total-Ratio__Simulations_Dummy__14_16_1000/sensitivity.csv", sep=';')

X = amsterdam_sensitivity_df[["TOTAL_RATIO"]]  # variables explicatives
y = amsterdam_sensitivity_df["NOX_RATIO"]

model = LinearRegression()
model.fit(X, y)

y_pred = model.predict(X)

# Plot
plt.scatter(X, y, color="blue", alpha=0.4,s=10, label="AmsterdamCity_7-9_Original Observations")
plt.plot(X, y_pred, color="steelblue", linewidth=0.8, label="AmsterdamCity_7-9_Original Régression linéaire")


plt.xlabel("TOTAL_RATIO")
plt.ylabel("NOX_RATIO")
plt.title("Sensibilité NOx en fonction du TOTAL_RATIO")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# comparatif CO2

matsim_sensitivity_df = pd.read_csv("../../outputs/Matsim_7-9/analysis/Matsim_7-9_Calibration__Variations__Total-Ratio__Sensitivity_Dummy__14_16_1000/sensitivity.csv", sep=';')

X = matsim_sensitivity_df[["TOTAL_RATIO"]]  # variables explicatives
y = matsim_sensitivity_df["CO2_RATIO"]

model = LinearRegression()
model.fit(X, y)

y_pred = model.predict(X)

# Plot
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color="red", alpha=0.4,s=10, label="Matsim_7-9_Calibration  Observations")
plt.plot(X, y_pred, color="darkred", linewidth=0.8, label="Matsim_7-9_Calibration Régression linéaire")


amsterdam_sensitivity_df = pd.read_csv("../../outputs/AmsterdamCity_7-9/analysis/AmsterdamCity_7-9_Original__Variations__Total-Ratio__Simulations_Dummy__14_16_1000/sensitivity.csv", sep=';')

X = amsterdam_sensitivity_df[["TOTAL_RATIO"]]  # variables explicatives
y = amsterdam_sensitivity_df["CO2_RATIO"]

model = LinearRegression()
model.fit(X, y)

y_pred = model.predict(X)

# Plot
plt.scatter(X, y, color="blue", alpha=0.4,s=10, label="AmsterdamCity_7-9_Original Observations")
plt.plot(X, y_pred, color="steelblue", linewidth=0.8, label="AmsterdamCity_7-9_Original Régression linéaire")


plt.xlabel("TOTAL_RATIO")
plt.ylabel("CO2_RATIO")
plt.title("Sensibilité NOx en fonction du TOTAL_RATIO")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

REFAIRE AVEC PLUS GRAND INTERVALLE

In [ ]:
matsim_sensitivity_df = pd.read_csv("../../outputs/Matsim_7-9/analysis/Matsim_7-9_Calibration__Variations__Total-Ratio__Sensitivity_Dummy__14_16_1000/sensitivity.csv", sep=';')
matsim_sensitivity_df